# 功能调试

## 概述

上一章我们开发了融合算子，本章进入算子开发的另一个关键环节：**调试调优**。本节聚焦功能调试——当算子运行结果不符合预期时，如何快速定位问题。

我们先看一个"出了问题"的算子。`./src/add_buggy.py` 是第2章Add算子的一个"变体"：代码可以正常编译、正常Launch、不报任何运行时错误，但**最终结果校验失败**：

```text
[INFO] start process sample add.
...
AssertionError: Tensor allclose check failed
```

算子没有崩溃、没有报错日志，只有最终的`torch.allclose`断言失败——问题出在哪里？核函数运行在Device侧，Host侧的`print`无法看到核内任何变量与数据。这就是功能调试的核心困难：**执行环境不可见**。

pyasc为Kernel侧代码提供了两个调试接口：

- **`asc.printf`**：打印标量与字符串信息（如block_idx、offset、循环变量），用来看"执行到了哪里、变量值是多少"；
- **`asc.dump_tensor`**：打印指定Tensor的数据内容（GM或UB），用来看"数据到底是什么"。

本节先讲清两个接口的原理与用法，再用它们完整定位`add_buggy.py`中的bug，最后总结通用调试方法论。

---
# 1. 环境准备

正式开始学习之前，先对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的准备。


In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
os.makedirs("Sources/06.02", exist_ok=True)
print("环境初始化完成")

---
# 2. 调试接口原理

### 2.1 Device侧打印机制

`asc.printf`与`asc.dump_tensor`都在**Kernel侧代码**（`@asc.jit`修饰的核函数内部）调用。它们的工作机制是：

1. 调试信息（格式化字符串/Tensor数据）在NPU执行时写入Device侧预留的输出缓冲区；
2. 算子执行结束后，CANN运行时把缓冲区内容回传到Host侧并打印到标准输出。

因此输出带有一个统一的**DumpHead头**，标识打印来源的核类型、block数量等信息，随后才是各打印点的输出内容。

### 2.2 与Host侧print的区别

| 对比项 | Host侧print | Device侧asc.printf/dump_tensor |
| --- | --- | --- |
| 调用位置 | launch函数前后 | 核函数内部任意位置 |
| 可见信息 | 只能看到Host变量 | 核内标量、Tensor数据 |
| 执行时机 | 核函数执行**之外** | 与算子指令同批次执行 |
| 性能影响 | 无 | 有（占用执行周期，调测期使用） |

### 2.3 输出格式解读

一次启用了printf与dump_tensor的运行，输出由三部分组成：DumpHead头、printf行、DumpTensor块：

<img src="./images/dump_output_format.png" alt="调试输出格式解读" width="900px">

- **DumpHead**：`opType`（算子类型）、`AIV-0`（打印来源核）、`block dim`（block总数）等元信息；
- **printf行**：按调用顺序输出的格式化文本；
- **DumpTensor块**：`desc`（打印点标识）、`addr`（地址）、`position`（GM/UB）、`dump_size`（元素数）+ 数据内容。


---
# 3. asc.printf详解

`asc.printf`用于在核函数中打印标量与字符串信息，适合观察**执行路径与变量取值**。

### 3.1 函数签名

```python
asc.printf(format_str, *args)
```

- `format_str`：格式化字符串，支持`%d`（整数）、`%f`（浮点）、`%s`（字符串）等占位符；
- `*args`：与占位符对应的变量。

### 3.2 使用示例

```python
asc.printf("Before calculating.\\n")
for i in range(TILE_NUM):
    asc.printf("current index is %d.\\n", i)
```

### 3.3 使用约束

| 约束 | 说明 |
| --- | --- |
| **换行需转义** | 源码中写作`\\n`（Python字符串中的字面`\n`两个字符），直接写`\n`不会换行 |
| **性能影响** | printf占用算子实际执行的周期，性能敏感场景调测结束后应移除 |
| **输出顺序** | 多核场景下各核输出交错，建议打印block_idx以便区分 |

> **注意：** 本教程Python源码示例中printf格式串的`\\n`，在pyasc编译处理后才成为Device侧的换行符——这与C语言printf的`\n`转义语义一致。


---
# 4. asc.dump_tensor详解

`asc.dump_tensor`用于打印指定Tensor的数据内容，适合核对**数据本身是否正确**（输入、中间结果、输出）。

### 4.1 函数签名

```python
asc.dump_tensor(tensor, desc_id, dump_size[, shape_info])
```

| 参数 | 类型 | 说明 |
| --- | --- | --- |
| `tensor` | GlobalTensor/LocalTensor | 待打印的Tensor，支持GM与UB位置 |
| `desc_id` | int | 打印点标识，用于区分多处dump的输出 |
| `dump_size` | int | 打印的元素个数（建议取小值如32，避免刷屏） |
| `shape_info` | ShapeInfo（可选） | 指定后按矩阵形态分行打印 |

### 4.2 输出解读

```text
DumpTensor: desc=0, addr=41200000, data_type=float32, position=GM, dump_size=32
[19.000000, 4.000000, 38.000000, ...]
```

- **desc**：即调用时传入的`desc_id`，混合多处dump时靠它区分数据来源；
- **position**：`GM`（GlobalMemory）或`UB`（UnifiedBuffer），标明数据所在存储位置；
- **data_type/dump_size**：数据类型与打印元素数。

### 4.3 ShapeInfo按矩阵形态打印

默认按一维打印。传入`ShapeInfo`可按指定行列分行输出，便于对照逻辑上的矩阵布局：

```python
tmp_array = asc.array(asc.uint32, [4, 16])       # 4行16列
tmp_shape_info = asc.ShapeInfo(tmp_array)
asc.dump_tensor(x_gm, 0, 32, tmp_shape_info)      # 按4x16矩阵形态打印
```

### 4.4 使用约束

- 与printf一样**有性能影响**，调测期使用，交付前移除；
- `dump_size`建议不超过32个元素：输出缓冲区有限，打印过多既刷屏也可能截断。


---
# 5. 调试实战：定位add_buggy.py的bug

现在用两个接口完整走一遍`add_buggy.py`的问题定位。回顾故障现象：编译运行无报错，`torch.allclose`校验失败。

### 5.1 复现故障

执行下方两个代码cell：先查看buggy版源码（重点看核函数中的offset计算），再运行复现故障——预期输出以`AssertionError: Tensor allclose check failed`结束，确认故障稳定复现。

### 5.2 提出假设并加printf取证

Add算子的结果由"每个核算哪段数据"决定，而这段数据由`offset = get_block_idx() * block_length`决定。**假设：切分参数计算有误，导致某些核读写了错误的数据段。**

取证手段：在每个核的核函数入口打印`block_idx`、`offset`、`block_length`三个值。`./src/add_debug.py`在buggy版基础上加入了两处调试语句（bug本身未修复）：

- **插入点1**（printf）：`set_global_buffer`之后，打印三个切分参数；
- **插入点2**（dump_tensor）：首次`data_copy`搬入x之后，仅在block 0的首个tile打印GM输入与UB数据各32个元素。

### 5.3 运行并解读printf输出

查看并运行调试版（5.2节后的两个代码cell），预期printf输出（节选）：

```text
block_idx=0, offset=0, block_length=8192.
block_idx=1, offset=512, block_length=8192.
block_idx=2, offset=1024, block_length=8192.
...
```

**解读**：总数据量65536，8核等分则每核block_length=8192，正确情况下offset应按`0, 8192, 16384, ...`递增；实际输出却是`0, 512, 1024, ...`——**offset只按tile_length（8192/8/2=512）递增**。也就是说，相邻核的数据段大量重叠：核1本应处理[8192, 16384)，实际处理的是[512, 8704)。

### 5.4 dump_tensor佐证

dump输出（desc=0为GM输入、desc=1为UB中搬入的数据）中，block 0的GM数据与其UB副本一致，说明**搬运本身没有问题**；结合printf已知的错误offset，可以确认：数据链路无损，问题只在于**每个核读写的起始位置错了**。

### 5.5 定位修复并复测

回到源码，offset计算处：

```python
offset = asc.get_block_idx() * tile_length  # BUG: 误用tile_length，应为block_length
```

修复为：

```python
offset = asc.get_block_idx() * block_length
```

同时`tile_length`的计算移回原位置。手工修复后重新运行`add_buggy.py`，输出`[INFO] Sample add run success.`，复测通过，调试闭环完成。

In [ ]:
# 5.1 查看buggy版源码（重点看核函数中的offset计算）
!cat ./src/add_buggy.py

In [ ]:
# 5.1 运行复现故障（预期allclose校验失败）
!python3 ./src/add_buggy.py -r NPU

In [ ]:
# 5.2/5.3 查看加了调试语句的完整源码（两处插入点见5.2节说明）
!cat ./src/add_debug.py

In [ ]:
# 5.3 运行调试版：观察printf的各核offset与dump_tensor的数据输出
!python3 ./src/add_debug.py -r NPU

---
# 6. 方法论小结

本节实战可以抽象为通用的六步调试流程：

<img src="./images/debug_workflow.png" alt="调试方法论流程" width="900px">

1. **现象**：明确故障表象（断言失败/数据错乱/NaN）；
2. **假设**：列出可能原因（切分错误/搬运错位/计算算子误用/同步缺失）；
3. **取证**：用printf看标量与路径，用dump_tensor看数据；
4. **定位**：对照预期值，锁定与假设不符的变量或数据；
5. **修复**：最小改动修复；
6. **复测**：验证修复且不引入新问题（必要时进入下一轮循环）。

### printf vs dump_tensor选型

| 要看什么 | 用什么 |
| --- | --- |
| 执行到哪了、循环/分支走向 | `asc.printf` |
| 标量参数对不对（offset/长度/索引） | `asc.printf` |
| 输入数据对不对（GM） | `asc.dump_tensor` |
| 中间/输出数据对不对（UB） | `asc.dump_tensor` |


---
# 7. 课后练习

### 选择题

**1.** 在pyasc核函数中使用`asc.printf`输出换行，Python源码里应如何书写？

- A. 直接写`\n`
- B. 写作`\\n`
- C. printf不支持换行
- D. 用`endl`

**2.** `asc.dump_tensor(x_gm, 0, 32)`中第二个参数`0`的作用是？

- A. 张量维度
- B. 数据类型编号
- C. 打印点标识（desc_id）
- D. 打印元素数

**3.** 排查"多核切分越界、不确定每个核算到哪段数据"的问题，应最先使用哪个工具？

- A. `asc.printf`打印offset与block_idx
- B. `asc.dump_tensor`打印全部数据
- C. msprof op
- D. 直接在Host侧加print

### 实践题

仿照`add_debug.py`，在**Compute阶段**（`asc.add`计算之后）对结果`z_local`增加一处`asc.dump_tensor`打印（建议desc_id=2、dump_size=32、仅block 0首拍打印），运行并验证：desc=2的输出是否等于对应`x_local+y_local`？

执行以下代码查看答案：


In [ ]:
!cat ./answer/06.02_answer.txt